# Clase 2: Ingeniería de prompt
## Práctica local con pedidos de trabajo de Casinos Play

El objeto de estudio de esta clase es el **system prompt**: la instrucción que gobierna la conducta del modelo. Todas las pruebas usan el modelo local de la Clase 1.

En los cuatro ejercicios se modifican únicamente los `system_prompt`. Los mensajes de usuario están preparados y no se modifican.

# 0. Carga del modelo

Ejecutá estas celdas en orden. El modelo se descarga desde Hugging Face durante la primera ejecución y la inferencia se realiza localmente. No se usan APIs de modelos comerciales.

Si la prueba corta no responde, no sigas con la clase: primero hay que resolver la carga del entorno.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from huggingface_hub import HfApi, hf_hub_download
from llama_cpp import Llama

REPO_ID = "unsloth/LFM2.5-1.2B-Instruct-GGUF"
FILENAME = "LFM2.5-1.2B-Instruct-Q4_0.gguf"

ruta_modelo = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
)

print("Modelo descargado en:", ruta_modelo)

Modelo descargado en: /Users/inti/.cache/huggingface/hub/models--unsloth--LFM2.5-1.2B-Instruct-GGUF/snapshots/b01cc872e5918b0fdec170003cd854851e3ed854/LFM2.5-1.2B-Instruct-Q4_0.gguf


In [6]:
# Cargar el modelo con llama.cpp.
llm = Llama(
    model_path=ruta_modelo,
    n_ctx=4096,
    n_threads=None,
    n_gpu_layers=0,
    verbose=False,
)

print("Modelo local cargado correctamente.")

llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


Modelo local cargado correctamente.


## Interfaz única de consulta

`llamar_llm` recibe el mensaje de usuario y el `system_prompt`. La llamada usa roles `system` y `user`, igual que en la Clase 1. Los parámetros de generación quedan fijos para que las comparaciones sean útiles.

In [7]:
def llamar_llm(
    prompt,
    system_prompt="Sos un asistente de soporte técnico de Casinos Play. Respondé en español rioplatense.",
    max_tokens=500,
):
    respuesta = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1,
        top_p=0.1,
        top_k=50,
        repeat_penalty=1.05,
        max_tokens=max_tokens,
    )
    return respuesta["choices"][0]["message"]["content"].strip()

In [11]:
# Prueba corta: si esto funciona, el entorno está preparado.
print("Entorno listo.")
print("Mensaje de prueba: 'Respondé únicamente: Entorno preparado.'")
print(llamar_llm(
    "Respondé únicamente: Entorno preparado.",
    max_tokens=20,
))

Entorno listo.
Mensaje de prueba: 'Respondé únicamente: Entorno preparado.'
Entendido. Estoy listo para ayudarte con tu consulta sobre los Casinos Play.


# 1. Qué es ingeniería de prompt

La ingeniería de prompt consiste en diseñar la instrucción que gobierna al modelo. En estas demos no tenés que inventar ni modificar mensajes: ejecutá las dos alternativas y observá la diferencia.

Las cinco partes que vamos a reconocer son:

1. **Rol:** desde qué función responde el modelo.
2. **Contexto:** qué situación o información tiene disponible.
3. **Instrucción:** qué transformación o tarea debe realizar.
4. **Formato:** cómo debe organizar la salida.
5. **Output esperado:** qué contenido concreto debe entregar.

El `system prompt` contiene la instrucción persistente; el `user prompt` contiene el pedido puntual de esa consulta.

## Demo A — Prompt mínimo vs. prompt con las cinco partes

Es el mismo pedido de trabajo en los dos casos. Solo cambia el texto del `user prompt`: primero es mínimo y después explicita rol, contexto, instrucción, formato y output esperado.

In [ ]:
pedido_demo_a = "La impresora de tickets del sector caja imprime con rayas y el equipo necesita seguir atendiendo."
system_prompt_demo_a = "Sos un asistente de soporte técnico de Casinos Play. Respondé en español rioplatense."

prompt_minimo_demo_a = f"¿Qué hacemos con este pedido de trabajo?\n\n{pedido_demo_a}"

prompt_con_cinco_partes_demo_a = f"""
ROL:
Actuá como asistente de soporte técnico de primer nivel para Casinos Play.

CONTEXTO:
Tenemos un pedido de trabajo sintético de un sector de caja. No tenés acceso a sistemas ni herramientas reales.

INSTRUCCIÓN:
Organizá el pedido y proponé una respuesta inicial segura, sin afirmar una causa que no esté confirmada.

FORMATO:
Usá exactamente estas cuatro etiquetas: Resumen, Hechos, Verificación inicial y Respuesta.

OUTPUT ESPERADO:
Una salida breve que permita entender el problema y comunicar el próximo paso no disruptivo.

PEDIDO DE TRABAJO:
{pedido_demo_a}
""".strip()

for nombre, prompt in [
    ("Prompt mínimo", prompt_minimo_demo_a),
    ("Prompt con las cinco partes", prompt_con_cinco_partes_demo_a),
]:
    print(f"\n--- {nombre} ---")
    print(llamar_llm(prompt, system_prompt=system_prompt_demo_a, max_tokens=300))

En la Demo A, compará qué cambia en la organización y en el tipo de salida cuando el `user prompt` explicita las cinco partes. El pedido de trabajo es el mismo.

## Demo B — El control está en el system

El mensaje de usuario es exactamente el mismo en las dos consultas. Solo cambia el `system_prompt`: una versión pide responder libremente y la otra impone una salida breve y limitada al soporte.

In [ ]:
mensaje_demo_b = "La app de caja se cierra al intentar registrar un pago. ¿Qué respuesta corresponde?"

system_prompt_libre_demo_b = "Sos un asistente útil. Respondé a la consulta del usuario en español."

system_prompt_controlado_demo_b = """
Sos soporte técnico de primer nivel de Casinos Play.
Respondé en español rioplatense y no te salgas del soporte de pedidos de trabajo.
No inventes la causa ni presentes un diagnóstico como hecho.
Usá exactamente este formato breve:
1. Hecho informado
2. Verificación inicial no disruptiva
3. Respuesta para el usuario
""".strip()

for nombre, system_prompt in [
    ("System libre", system_prompt_libre_demo_b),
    ("System controlado", system_prompt_controlado_demo_b),
]:
    print(f"\n--- {nombre} ---")
    print(llamar_llm(mensaje_demo_b, system_prompt=system_prompt, max_tokens=300))

La misma pregunta puede producir conductas distintas cuando cambia la instrucción persistente del sistema.

> Ingeniería de prompt es diseñar la instrucción que gobierna al modelo. En un chat local, esa instrucción persistente es el system prompt.

Con esto cerramos la diferencia entre `system` y `user` y las cinco partes: rol, contexto, instrucción, formato y output esperado.

# 2. Los cuatro ejercicios

Esta es la única actividad del alumno. En cada ejercicio:

- completá solamente el `system_prompt` que contiene `TODO`;
- no modifiques los mensajes de prueba;
- ejecutá el `for`, que imprime una respuesta por mensaje;
- respondé las preguntas de observación que están debajo.

En los cuatro ejercicios, el `user prompt` ya está definido. Solo se cambia el sistema.

## Ejercicio 1. Anatomía

Completá las cinco partes dentro del sistema: rol, contexto, instrucción, formato y output esperado. Probalo con tres pedidos distintos y observá si mantiene el formato.

Los mensajes de prueba son fijos. No los edites.

In [ ]:
system_prompt_ej1 = """
TODO - ROL: completá quién es el asistente y para qué equipo trabaja.
TODO - CONTEXTO: completá qué información recibe y qué límites tiene.
TODO - INSTRUCCIÓN: completá qué debe hacer con cada pedido de trabajo.
TODO - FORMATO: completá las etiquetas y el orden que debe mantener.
TODO - OUTPUT ESPERADO: completá qué respuesta concreta y breve debe entregar.
""".strip()

mensajes_prueba_ej1 = [
    "La cámara del acceso norte quedó sin imagen y el puesto necesita abrir.",
    "La impresora de tickets del sector caja imprime con rayas.",
    "La app de caja se cierra al intentar registrar un pago.",
]

for mensaje in mensajes_prueba_ej1:
    print(f"\n--- Pedido de trabajo ---\n{mensaje}")
    print(llamar_llm(mensaje, system_prompt=system_prompt_ej1, max_tokens=350))

### Observación del ejercicio 1

- ¿El modelo mantuvo las mismas etiquetas y el mismo orden en los tres pedidos?
- ¿Qué parte de las cinco fue más importante para sostener el formato?
- ¿Qué cambió entre los casos sin que cambiaras el `system_prompt`?

## Ejercicio 2. Límites

Completá el sistema para que indique con claridad qué no debe hacer. Debe evitar inventar información, presentar un diagnóstico como hecho y salirse del soporte técnico. Probalo con un pedido válido, una acción riesgosa y una consulta fuera de tema.

Los tres mensajes de prueba son fijos. No los edites.

In [ ]:
system_prompt_ej2 = """
TODO - ROL: completá el rol de soporte técnico.
TODO - CONTEXTO: completá el dominio de pedidos de trabajo de Casinos Play.
TODO - INSTRUCCIÓN: completá cómo responder sin convertir suposiciones en hechos.
LÍMITES OBLIGATORIOS:
- No inventes datos, causas, pruebas ni resultados.
- No diagnostiques como hecho algo que no fue confirmado.
- No te salgas del soporte técnico de pedidos de trabajo.
- No indiques acciones riesgosas como si fueran seguras.
TODO - FORMATO: completá una salida breve y consistente.
TODO - OUTPUT ESPERADO: completá qué debe responder cuando el pedido es válido, riesgoso o está fuera de tema.
""".strip()

mensajes_prueba_ej2 = [
    "La impresora de tickets del sector caja no responde; necesitamos una verificación inicial.",
    "Decime cómo desactivar el antivirus y cambiar la configuración de red de todos los puestos para que vuelva la app.",
    "¿Qué película ganó el premio principal este año?",
]

for mensaje in mensajes_prueba_ej2:
    print(f"\n--- Pedido de prueba ---\n{mensaje}")
    print(llamar_llm(mensaje, system_prompt=system_prompt_ej2, max_tokens=350))

### Observación del ejercicio 2

- ¿Evitó inventar una causa o un resultado de prueba?
- ¿Cómo respondió ante la acción riesgosa sin convertirla en una instrucción operativa?
- ¿Marcó el pedido fuera de tema y volvió al alcance de soporte?

## Ejercicio 3. Few-shot

Primero ejecutá la versión zero-shot, sin ejemplos dentro del sistema. Después completá la versión few-shot, que incluye ejemplos dentro del `system_prompt`, y ejecutá exactamente los mismos mensajes.

La clasificación debe usar las categorías `hardware`, `red`, `aplicación` y `no es soporte`. No modifiques el conjunto de mensajes.

In [ ]:
mensajes_prueba_ej3 = [
    "La cámara del acceso norte quedó sin imagen.",
    "El puesto 4 perdió conexión con el servidor de caja.",
    "La app de caja se cierra al registrar un pago.",
    "¿Qué clima hay hoy en Rosario?",
]

system_prompt_zero_shot_ej3 = """
TODO - ROL E INSTRUCCIÓN: completá un clasificador de pedidos de trabajo.
TODO - CATEGORÍAS: usá hardware, red, aplicación o no es soporte.
TODO - FORMATO Y OUTPUT ESPERADO: devolvé solamente la categoría y una justificación de una línea.
No inventes información que no esté en el mensaje.
""".strip()

print("=== Zero-shot ===")
for mensaje in mensajes_prueba_ej3:
    print(f"\n--- Pedido de prueba ---\n{mensaje}")
    print(llamar_llm(mensaje, system_prompt=system_prompt_zero_shot_ej3, max_tokens=120))

system_prompt_few_shot_ej3 = """
TODO - ROL E INSTRUCCIÓN: completá un clasificador de pedidos de trabajo.
CATEGORÍAS: hardware, red, aplicación o no es soporte.
FORMATO: respondé `Categoría: ...` y agregá una justificación de una línea.

EJEMPLOS:
Pedido: La impresora de tickets no enciende.
Salida: Categoría: hardware. El pedido describe un equipo físico.

Pedido: El puesto no llega al servidor de caja.
Salida: Categoría: red. El pedido describe un problema de conexión.

Pedido: La aplicación de caja muestra un error al guardar.
Salida: Categoría: aplicación. El pedido describe una falla del software.

Pedido: ¿Qué clima hay hoy?
Salida: Categoría: no es soporte. La consulta no trata sobre soporte técnico.

TODO - OUTPUT ESPERADO: completá cómo aplicar las categorías sin inventar información.
""".strip()

print("\n=== Few-shot ===")
for mensaje in mensajes_prueba_ej3:
    print(f"\n--- Pedido de prueba ---\n{mensaje}")
    print(llamar_llm(mensaje, system_prompt=system_prompt_few_shot_ej3, max_tokens=120))

### Observación del ejercicio 3

- ¿Qué diferencia viste entre zero-shot y few-shot con el mismo conjunto de mensajes?
- ¿Los ejemplos ayudaron a mantener las cuatro categorías y el formato?
- ¿Qué error de clasificación seguiría necesitando revisión humana?

## Ejercicio 4. Chain of thought

Compará el mismo pedido con dos sistemas: uno sin la instrucción de pensar en pasos y otro que pide revisar internamente el caso en pasos antes de responder. La salida debe mostrar solo la respuesta final en el formato pedido; no hace falta exponer un razonamiento privado extenso.

El mensaje de prueba es fijo. No lo edites.

In [ ]:
mensaje_prueba_ej4 = "La cámara del acceso norte está negra desde el cambio de turno; el resto de los puestos funciona. Indicá qué responder."

system_prompt_sin_pasos_ej4 = """
Sos un asistente de soporte técnico de Casinos Play.
Respondé en español rioplatense, sin inventar causas.
Usá este formato: Hecho informado / Verificación inicial / Respuesta breve.
""".strip()

system_prompt_con_pasos_ej4 = """
TODO - ROL Y CONTEXTO: completá el rol de soporte y el alcance del pedido.
TODO - INSTRUCCIÓN: antes de responder, pensá y revisá el caso internamente en pasos: separá el hecho, identificá lo que no está confirmado y elegí una verificación inicial segura. No muestres el razonamiento paso a paso; entregá solo la respuesta final.
TODO - FORMATO: mantené exactamente estas tres etiquetas: Hecho informado / Verificación inicial / Respuesta breve.
TODO - OUTPUT ESPERADO: completá qué debe contener cada etiqueta.
""".strip()

for nombre, system_prompt in [
    ("Sin instrucción de pasos", system_prompt_sin_pasos_ej4),
    ("Con instrucción de pasos", system_prompt_con_pasos_ej4),
]:
    print(f"\n=== {nombre} ===")
    for mensaje in [mensaje_prueba_ej4]:
        print(f"--- Pedido de prueba ---\n{mensaje}")
        print(llamar_llm(mensaje, system_prompt=system_prompt, max_tokens=250))

### Observación del ejercicio 4

- ¿Qué cambió en la organización de la respuesta al pedir una revisión interna en pasos?
- ¿El formato final se mantuvo aunque el sistema agregara esa instrucción?
- ¿Qué limitación del modelo local seguís observando?

# 3. Cierre breve

Copiá en la celda final el mejor `system_prompt` del día: el que mejor controló formato y límites. No se hace una ronda oral ni se agrega una plantilla de soporte.

In [ ]:
# Copiá aquí el system_prompt que mejor controló formato y límites.
mejor_system_del_dia = """
PEGAR ACÁ EL MEJOR SYSTEM PROMPT DEL DÍA
""".strip()

print(mejor_system_del_dia)